# Reconstruct Markowitz from a one-factor model

The tangency book inverts a mean and a covariance. Filling those from a trailing window is too noisy: a two-week slide reorders the sample means by several annualized percent, and the inverse turns residual covariances into leveraged long/short bets.

The first-principles matrices are not a dense N x N sample. They come from a one-factor world (formulas below are PNGs, because Cursor does not render MathJax):

![One-factor model: returns, covariance, mean, tangency weights](figures/eq_model.png)

That is 2N+2 structural parameters, not N(N+1)/2 covariances. Residual alphas and residual covariances are estimation error. They are not an investment opportunity.

`reconstruct_factor_moments` rebuilds both matrices from the spiked eigenstructure of the window: keep the market spike (lambda_1, q_1), replace the bulk with its average eigenvalue, project the sample mean onto that beta. The inverse is closed form (Woodbury), so there is no 1/lambda_min noise left to invert.

In [1]:
from pathlib import Path

from pytorch_katas.portfolio import (
    TRADING_DAYS,
    TWO_WEEKS,
    allocate_window_pair,
    cov_condition_number,
    mean_shift_std,
    reconstruct_factor_moments,
    save_equation_plates,
    save_sensitivity_figures,
    sample_moments,
    simulate_equity_universe,
    summarize_pair,
)
from pytorch_katas.settings import DATA_DIR

# Rasterize identities next to this notebook so the markdown images above resolve.
NOTEBOOK_FIG = Path.cwd() / "figures"
if not (NOTEBOOK_FIG.parent / "markowitz_sensitivity.ipynb").exists():
    NOTEBOOK_FIG = Path.cwd() / "notebooks" / "portfolio" / "figures"
save_equation_plates(NOTEBOOK_FIG)

{'eq_model': PosixPath('/workspace/notebooks/portfolio/figures/eq_model.png'),
 'eq_woodbury': PosixPath('/workspace/notebooks/portfolio/figures/eq_woodbury.png'),
 'eq_mean_shift': PosixPath('/workspace/notebooks/portfolio/figures/eq_mean_shift.png')}

## Why the sample matrices are the wrong object

Sliding a window of length T by h days moves a sample mean by:

![Std of the change in sample mean when the window slides](figures/eq_mean_shift.png)

For daily vol 1.5%, T = 252, h = 10 that is already about 7% annualized — the same order as the premia you are trying to rank.

In [2]:
daily_vol = 0.015
std_daily = mean_shift_std(daily_vol, TRADING_DAYS, TWO_WEEKS)
print(f"std of sample d-mu (annualized): {std_daily * TRADING_DAYS:.2%}")

std of sample d-mu (annualized): 6.71%


## Rebuild Sigma and mu, then invert those

Twelve names, one market factor. A year-long window, then the same window ten trading days later. Compare the raw sample book to the book that inverts the reconstructed matrices.

In [3]:
universe = simulate_equity_universe(n_days=750, n_assets=12, seed=0)
pair = allocate_window_pair(universe.returns, start=200)
print(summarize_pair(pair, universe.names, universe=universe))

Window 252 days, shifted by 10 trading days (~2 weeks).
Condition number of sample Σ̂:        54.3
Condition number of reconstructed Σ: 32.5
Largest |Δμ| sample / factor (ann.): 7.08% / 0.12%
Cross-sectional std of Δμ sample / factor (ann.): 3.45% / 0.03%
Relative Frobenius error vs true Σ, sample:        12.57%
Relative Frobenius error vs true Σ, reconstructed: 12.33%

rule           turnover  max |w| A  max |w| B   max |Δw|
tangency        302.6%     334.8%     398.0%      98.1%
gmv              11.5%      45.4%      49.9%       4.5%
ridge           104.5%     132.9%     133.8%      42.3%
shrink           25.5%      55.2%      52.9%      10.5%
factor            0.4%      10.4%      10.3%       0.2%
long_only         0.0%     100.0%     100.0%       0.0%
equal             0.0%       8.3%       8.3%       0.0%

Annualized means, sample vs reconstructed (window A → B):
  A01: sample  -7.05% →  -8.85% (Δ  -1.81%)   factor   2.98% →   2.99% (Δ  +0.02%)
  A02: sample  -1.52% →   4.11% (Δ  

The reconstruction of one window, written out. After scaling the factor to unit variance, Sigma is beta beta^T + sigma_bar^2 I. mu is the projection of the sample mean onto beta.

In [4]:
window = universe.returns[200:452]
fit = reconstruct_factor_moments(window)
mu_hat, cov_hat = sample_moments(window)
print(f"price of risk lambda (daily): {fit.price_of_risk:.5f}")
print(f"idio variance:                {fit.idio_var[0]:.2e}")
print(f"true idio variance:           {universe.idio_vol[0] ** 2:.2e}")
print(f"sample cond(Sigma):           {cov_condition_number(cov_hat):.1f}")
print(f"reconstructed cond(Sigma):    {cov_condition_number(fit.cov):.1f}")
print("mu = lambda * beta (annualized):", (fit.mu * TRADING_DAYS).round(4))

price of risk lambda (daily): 0.01227
idio variance:                6.30e-05
true idio variance:           6.40e-05
sample cond(Sigma):           54.3
reconstructed cond(Sigma):    32.5
mu = lambda * beta (annualized): [0.0298 0.03   0.0351 0.0329 0.0367 0.0387 0.0376 0.0416 0.0449 0.0468
 0.0477 0.0489]


## What this does to the book

The inverse of a rank-one-plus-isotropic covariance is Woodbury, not a noisy eigendecomposition:

![Woodbury inverse of the reconstructed covariance](figures/eq_woodbury.png)

Because mu lives in span{beta}, the tangency weights are a mild tilt along beta. They cannot load the residual subspace that a two-week slide keeps reshuffling.

Ledoit-Wolf plus a James-Stein shrink of the means is a statistical cousin of the same idea. The factor rebuild is the model those shrinkages are approximating.

In [5]:
fig_dir = Path(DATA_DIR) / "portfolio_figures"
paths = save_sensitivity_figures(universe, fig_dir)
for key, path in paths.items():
    print(f"{key:16s} {path}")

covariance       /workspace/data/portfolio_figures/covariance_reconstruction.png
moments          /workspace/data/portfolio_figures/factor_moment_reconstruction.png
weights          /workspace/data/portfolio_figures/window_shift_weights.png
rolling          /workspace/data/portfolio_figures/rolling_allocation_heatmap.png
eq_model         /workspace/data/portfolio_figures/eq_model.png
eq_woodbury      /workspace/data/portfolio_figures/eq_woodbury.png
eq_mean_shift    /workspace/data/portfolio_figures/eq_mean_shift.png


Plots written to `data/portfolio_figures/` (and the equation plates next to this notebook):

- `covariance_reconstruction.png` — sample Sigma, reconstructed factor Sigma, true DGP Sigma.
- `factor_moment_reconstruction.png` — spectra (keep the spike, flatten the bulk) and mu = lambda beta versus the sample means.
- `window_shift_weights.png` — invert the sample matrices vs invert the reconstructed ones, ten days apart.
- `rolling_allocation_heatmap.png` — the same contrast over the whole sample.

The allocation stopped jumping because the matrices stopped being a dump of every residual in the window.